# Consumer Complaints ML Training, EDA, and Model Review

This notebook is fully self-contained and runs directly in Databricks.

It validates the managed feature table, reviews class imbalance, uses Random Forest importance to select the strongest features, and then compares the available tree-based models:
- Decision Tree
- Random Forest
- XGBoost

Model training runs on pandas/scikit-learn/XGBoost rather than Spark ML: this workspace's serverless
compute does not support classic `pyspark.ml` reliably (constructor whitelisting and Spark Connect ML
session cache limits), and the reduced, already-aggregated feature table here is small enough to train
comfortably on the driver. Spark is still used for reading the managed Gold-layer feature table and for
all EDA aggregations, since those scale over the full table.

No linear model is used in this notebook.

## Step 1: Imports, constants, and Spark session

In [ ]:
import json  # Serialize serving_metadata to a UC Volume file.
import logging  # Keep notebook execution easy to follow during long runs.

import numpy as np  # Numeric arrays for model inputs.
import pandas as pd  # Pandas is the training data representation for this notebook.
from sklearn.ensemble import RandomForestClassifier  # Random forest classifier and feature selector.
from sklearn.calibration import CalibratedClassifierCV, calibration_curve  # Fix probability distortion introduced by class weighting.
from sklearn.metrics import average_precision_score, fbeta_score, precision_recall_curve, precision_score, recall_score, roc_auc_score  # Imbalance-aware evaluation metrics and threshold selection.
from sklearn.model_selection import GridSearchCV, KFold, train_test_split  # Optional hyperparameter tuning; K-fold target encoding for TRAIN; calibration/threshold-selection split.
import matplotlib.pyplot as plt  # Reliability curve plots.
from sklearn.tree import DecisionTreeClassifier  # Decision tree classifier requested for this project.
from pyspark.sql import SparkSession  # Main Spark entry point, used for reading/writing managed tables.
from pyspark.sql import functions as F  # Spark SQL helpers used for EDA aggregations.

try:
    from xgboost import XGBClassifier  # Gradient-boosted tree classifier.
    XGBOOST_AVAILABLE = True  # Track whether XGBoost can be included in this runtime.
    XGBOOST_IMPORT_ERROR = None
except Exception as xgboost_import_error:  # noqa: BLE001 - report whatever the real failure is (missing native lib, version conflict, etc.), not just ImportError.
    XGBClassifier = None  # Keep the notebook runnable even when XGBoost is unavailable.
    XGBOOST_AVAILABLE = False  # Free or lightweight runtimes may not ship with xgboost.
    XGBOOST_IMPORT_ERROR = xgboost_import_error


spark = SparkSession.builder.getOrCreate()  # Reuse the active Databricks Spark session.

CATALOG = "fintech_lakehouse_dev"  # Development catalog for this project.
GOLD_SCHEMA = "gold"  # Gold schema that stores the feature table.
MONITORING_SCHEMA = "monitoring"  # Monitoring schema for model outputs.

FEATURE_TABLE = f"{CATALOG}.{GOLD_SCHEMA}.consumer_complaints_timely_response_features"  # Managed feature table created by the features notebook.
METRICS_TABLE = f"{CATALOG}.{MONITORING_SCHEMA}.consumer_complaints_timely_response_model_metrics"  # Table that stores model metrics.
PREDICTIONS_TABLE = f"{CATALOG}.{MONITORING_SCHEMA}.consumer_complaints_timely_response_test_predictions"  # Table that stores held-out predictions.

LABEL_COLUMN = "label_untimely_response"  # Binary target label.
WEIGHT_COLUMN = "class_weight"  # Weight column used to handle class imbalance.
SPLIT_COLUMN = "dataset_split"  # Deterministic split column from feature engineering.
MODEL_NAME_COLUMN = "model_name"  # Model identifier persisted with outputs.
SELECTED_FEATURE_RANK_COLUMN = "selected_feature_rank"  # Rank assigned during feature selection.

TRAIN_SPLIT_LABEL = "TRAIN"  # Training split label.
VALIDATION_SPLIT_LABEL = "VALIDATION"  # Validation split label.
TEST_SPLIT_LABEL = "TEST"  # Test split label.

DECISION_TREE_MODEL_NAME = "decision_tree"  # Decision tree benchmark name.
RANDOM_FOREST_MODEL_NAME = "random_forest"  # Random forest benchmark name.
XGBOOST_MODEL_NAME = "spark_xgboost"  # Keep the historical model name so downstream dashboards/queries do not break.

CATEGORICAL_FEATURE_COLUMNS = [
    "product",  # High-level complaint product category.
    "sub_product",  # More granular product detail.
    "company",  # Company handling the complaint.
    "state",  # Complaint geography.
    "issue",  # Main complaint issue bucket.
    "sub_issue",  # Finer complaint issue category.
    "submitted_via",  # Intake channel.
    "zip_code_prefix3",  # Coarse locality signal.
    "received_month_name",  # Seasonality label.
    "received_day_name",  # Weekday label.
]

# TODO: "has_consumer_narrative" is a binary presence flag only; the actual narrative text is
# not used as a feature anywhere in this pipeline. Real text features (e.g. embeddings from a
# pretrained transformer) are a plausible source of additional signal but are out of scope for
# this iteration - they need new plumbing in Silver/Gold to retain the raw text, plus new
# feature-engineering and training work. Do not add this without first scoping that work properly.
NUMERIC_FEATURE_COLUMNS = [
    "complaint_received_year",  # Intake year.
    "complaint_received_quarter",  # Intake quarter.
    "complaint_received_month",  # Intake month.
    "complaint_received_day",  # Day of month.
    "complaint_received_day_of_week",  # Day-of-week index.
    "received_is_weekend",  # Weekend flag.
    "has_consumer_narrative",  # Narrative-present indicator.
    "has_tags",  # Tags-present indicator.
]

INDEXED_CATEGORICAL_FEATURE_COLUMNS = [f"{column_name}_indexed" for column_name in CATEGORICAL_FEATURE_COLUMNS]  # Indexed versions of the categorical columns.
MODEL_INPUT_COLUMNS = INDEXED_CATEGORICAL_FEATURE_COLUMNS + NUMERIC_FEATURE_COLUMNS  # These 18 fields are the full candidate input set.
TOP_FEATURE_COUNT = 10  # Keep only the top 10 features after Random Forest importance ranking.

PREDICTION_OUTPUT_COLUMNS = [
    "complaint_id", "product", "sub_product", "company", "state",
    "issue", "sub_issue", "submitted_via", "complaint_received_date",
]  # Reporting columns carried through to the persisted predictions table.

logging.basicConfig(level=logging.INFO, format="%(asctime)s %(levelname)s %(message)s")  # Standardised log format for easier debugging.
LOGGER = logging.getLogger("consumer_complaints_ml_train_notebook")  # Notebook-specific logger.

print("Feature table:", FEATURE_TABLE)
print("Full candidate X_train feature count:", len(MODEL_INPUT_COLUMNS))
print("y_train label count:", 1)
print("XGBoost available in this runtime:", XGBOOST_AVAILABLE)
if not XGBOOST_AVAILABLE:
    print('XGBoost import failed:', repr(XGBOOST_IMPORT_ERROR))  # Surface the real cause instead of silently dropping the model.

# Persist this diagnostic to a small table so it can be queried directly (e.g. via a SQL warehouse)
# instead of relying on someone finding and copying this print statement out of the notebook UI.
from pyspark.sql.types import BooleanType, StringType, StructField, StructType  # Explicit schema: an all-None column otherwise fails type inference.
diagnostics_schema = StructType([
    StructField('xgboost_available', BooleanType(), False),
    StructField('xgboost_import_error', StringType(), True),
])
spark.sql(f"CREATE SCHEMA IF NOT EXISTS {CATALOG}.{MONITORING_SCHEMA}")
spark.createDataFrame(
    [(bool(XGBOOST_AVAILABLE), str(XGBOOST_IMPORT_ERROR) if XGBOOST_IMPORT_ERROR is not None else None)],
    schema=diagnostics_schema,
).withColumn('_checked_at', F.current_timestamp()).write.format('delta').mode('overwrite').saveAsTable(f'{CATALOG}.{MONITORING_SCHEMA}.consumer_complaints_ml_runtime_diagnostics')


## Step 2: Load and validate the managed feature table

In [ ]:
if not spark.catalog.tableExists(FEATURE_TABLE):  # Fail fast if feature engineering has not been run yet.
    raise RuntimeError(f"ML feature table does not exist: {FEATURE_TABLE}")

feature_dataframe = spark.table(FEATURE_TABLE)  # Read the saved feature table.
feature_row_count = feature_dataframe.count()  # Count rows for validation and reporting.

if feature_row_count == 0:
    raise RuntimeError(f"ML feature table is empty: {FEATURE_TABLE}")

split_counts = {row[SPLIT_COLUMN]: row['row_count'] for row in feature_dataframe.groupBy(SPLIT_COLUMN).count().withColumnRenamed('count', 'row_count').collect()}  # Summarise split sizes for quick checks.

for required_split in [TRAIN_SPLIT_LABEL, VALIDATION_SPLIT_LABEL, TEST_SPLIT_LABEL]:
    if split_counts.get(required_split, 0) == 0:  # All three splits must exist before training begins.
        raise RuntimeError(f"ML feature validation failed: required split {required_split} is empty.")

LOGGER.info("Validated ML feature table %s with %s rows.", FEATURE_TABLE, f"{feature_row_count:,}")
display(feature_dataframe.limit(20))  # Preview the model-ready rows.

## Step 3: Review split balance and class imbalance

In [ ]:
display(feature_dataframe.groupBy(SPLIT_COLUMN).count().orderBy(SPLIT_COLUMN))  # Check train, validation, and test volumes.
display(feature_dataframe.groupBy(LABEL_COLUMN).count().orderBy(LABEL_COLUMN))  # Measure overall class imbalance.
display(feature_dataframe.groupBy(SPLIT_COLUMN, LABEL_COLUMN).count().orderBy(SPLIT_COLUMN, LABEL_COLUMN))  # Confirm every split contains both classes.

## Step 4: Correlation matrix for numeric features

In [ ]:
numeric_columns = NUMERIC_FEATURE_COLUMNS + [LABEL_COLUMN]  # Include the label to inspect coarse feature-to-target relationships.

# Pairwise DataFrame.stat.corr calls run entirely as Spark aggregates - no local collection of raw rows required.
correlation_rows = [
    (
        row_column,
        *[float(feature_dataframe.stat.corr(row_column, column_column, method='pearson')) for column_column in numeric_columns],
    )
    for row_column in numeric_columns
]
corr_matrix_df = spark.createDataFrame(correlation_rows, ['feature_name'] + numeric_columns)  # Queryable Pearson correlation matrix.

display(corr_matrix_df)  # Display the matrix compactly.
print('Correlation feature order:', numeric_columns)

## Step 5: Load the model-ready columns into pandas and create X_train / y_train

In [ ]:
# Pull only the columns training needs. The Gold feature table is complaint-level and reduced to
# these candidate inputs, so this comfortably fits in driver memory as a pandas DataFrame.
pandas_columns = list(dict.fromkeys(
    ['complaint_id', SPLIT_COLUMN, LABEL_COLUMN] + CATEGORICAL_FEATURE_COLUMNS + NUMERIC_FEATURE_COLUMNS + PREDICTION_OUTPUT_COLUMNS
))  # De-duplicate while preserving order (categorical columns overlap with the reporting columns).

import time  # Used for a small backoff between toPandas retry attempts.

def collect_to_pandas_with_retry(spark_dataframe, attempts=3, initial_backoff_seconds=10):
    # toPandas() over Spark Connect has occasionally hit a transient internal server error on this
    # workspace; retrying a couple of times with backoff is cheap insurance against that.
    last_error = None
    for attempt in range(1, attempts + 1):
        try:
            return spark_dataframe.toPandas()
        except Exception as collect_error:  # noqa: BLE001 - deliberately broad to retry any transient Connect failure.
            last_error = collect_error
            LOGGER.warning('toPandas attempt %s/%s failed: %s', attempt, attempts, collect_error)
            if attempt < attempts:
                time.sleep(initial_backoff_seconds * attempt)
    raise last_error

# Collect one split at a time, capped to a representative sample per split. Multi-million-row
# toPandas() transfers over Spark Connect have consistently failed on this workspace (a resource/size
# limit, not a transient blip - retries alone did not help). A baseline tree-based model does not need
# every row: a few hundred thousand rows per split is standard practice and plenty for stable training.
MAX_ROWS_PER_SPLIT = 500_000  # Cap collected rows per split to keep the toPandas() transfer reliable.

split_pandas_frames = []
for split_label in [TRAIN_SPLIT_LABEL, VALIDATION_SPLIT_LABEL, TEST_SPLIT_LABEL]:
    split_spark_df = feature_dataframe.filter(F.col(SPLIT_COLUMN) == split_label).select(*pandas_columns)
    split_row_count = split_spark_df.count()  # Cheap Spark aggregate, not a full collect.
    if split_row_count > MAX_ROWS_PER_SPLIT:
        sample_fraction = min(1.0, MAX_ROWS_PER_SPLIT / split_row_count)
        split_spark_df = split_spark_df.sample(fraction=sample_fraction, seed=42)  # Reproducible random sample.
        LOGGER.info('%s split has %s rows; sampling down to ~%s rows (fraction=%.4f).', split_label, f'{split_row_count:,}', f'{MAX_ROWS_PER_SPLIT:,}', sample_fraction)
    split_pandas_frames.append(collect_to_pandas_with_retry(split_spark_df))
    LOGGER.info('Collected %s split into pandas: %s rows.', split_label, f'{len(split_pandas_frames[-1]):,}')

feature_pandas_df = pd.concat(split_pandas_frames, ignore_index=True)  # Bring the reduced feature table to the driver, split by split.

LOGGER.info("Collected %s rows x %s columns into pandas for training.", f"{len(feature_pandas_df):,}", len(pandas_columns))

training_only_pdf = feature_pandas_df[feature_pandas_df[SPLIT_COLUMN] == TRAIN_SPLIT_LABEL]  # Use only the training split to derive class weights.
class_counts = training_only_pdf[LABEL_COLUMN].value_counts().to_dict()  # Count each class in the training split only.

negative_count = int(class_counts.get(0, 0))  # Timely complaints form the majority negative class.
positive_count = int(class_counts.get(1, 0))  # Untimely complaints form the minority positive class.

if negative_count == 0 or positive_count == 0:
    raise RuntimeError('ML training failed: both label classes must be present in the training split.')

positive_weight = negative_count / positive_count  # Up-weight the rare NO class so the models pay attention to it.
feature_pandas_df[WEIGHT_COLUMN] = np.where(feature_pandas_df[LABEL_COLUMN] == 1, positive_weight, 1.0)  # Add the class weight used by all three models.

# ASSUMPTION (opinion): OrdinalEncoder was replaced with target encoding. OrdinalEncoder assigns
# arbitrary integer codes (0, 1, 2, ...) to categories with no real order (e.g. company names),
# which falsely implies category 47 is 'closer to' category 48 than to category 2. Tree models can
# partially work around this via repeated splits, but it caps how much signal they can extract -
# and company/product were the top two most important features, so this plausibly costs real signal.
#
# Target encoding replaces each category with its (smoothed) historical positive rate instead. To
# avoid leaking each row's own label into its own encoded value, TRAIN rows are encoded out-of-fold
# (5-fold: each fold's categories are encoded using only the OTHER folds' statistics); VALIDATION
# and TEST are encoded using the full TRAIN split's statistics, since they are genuinely held out.
# Smoothing (smoothing=20) pulls rare categories' estimates toward the global TRAIN rate, so a
# company with only 2 complaints does not get an extreme, noisy encoded value.
TARGET_ENCODING_SMOOTHING = 20
TARGET_ENCODING_FOLDS = 5

global_train_positive_rate = training_only_pdf[LABEL_COLUMN].mean()

def smoothed_category_means(fit_pdf, column_name):
    stats = fit_pdf.groupby(column_name)[LABEL_COLUMN].agg(['mean', 'count'])
    smoothed = (stats['count'] * stats['mean'] + TARGET_ENCODING_SMOOTHING * global_train_positive_rate) / (stats['count'] + TARGET_ENCODING_SMOOTHING)
    return smoothed

train_pdf = feature_pandas_df[feature_pandas_df[SPLIT_COLUMN] == TRAIN_SPLIT_LABEL].copy()  # X_train and y_train come from this split.
validation_pdf = feature_pandas_df[feature_pandas_df[SPLIT_COLUMN] == VALIDATION_SPLIT_LABEL].copy()  # Validation split for model comparison.
test_pdf = feature_pandas_df[feature_pandas_df[SPLIT_COLUMN] == TEST_SPLIT_LABEL].copy()  # Held-out test split for final reporting.

kfold = KFold(n_splits=TARGET_ENCODING_FOLDS, shuffle=True, random_state=42)
categorical_encoding_maps = {}  # column_name -> {category: smoothed_rate}, needed to encode new raw complaints at serving time.
for column_name, indexed_column_name in zip(CATEGORICAL_FEATURE_COLUMNS, INDEXED_CATEGORICAL_FEATURE_COLUMNS, strict=True):
    encoded_train_values = np.empty(len(train_pdf), dtype=float)
    for fit_positions, transform_positions in kfold.split(train_pdf):
        fit_fold_pdf = train_pdf.iloc[fit_positions]
        transform_fold_pdf = train_pdf.iloc[transform_positions]
        fold_means = smoothed_category_means(fit_fold_pdf, column_name)
        encoded_train_values[transform_positions] = transform_fold_pdf[column_name].map(fold_means).fillna(global_train_positive_rate).to_numpy()
    train_pdf[indexed_column_name] = encoded_train_values

    full_train_means = smoothed_category_means(train_pdf, column_name)  # Fit on ALL of TRAIN for VALIDATION/TEST - they are genuinely held out, so no fold-splitting needed.
    validation_pdf[indexed_column_name] = validation_pdf[column_name].map(full_train_means).fillna(global_train_positive_rate)
    test_pdf[indexed_column_name] = test_pdf[column_name].map(full_train_means).fillna(global_train_positive_rate)
    categorical_encoding_maps[column_name] = full_train_means.to_dict()

print('Training class weight for NO rows:', round(positive_weight, 4))
print('X_train raw feature count:', len(CATEGORICAL_FEATURE_COLUMNS) + len(NUMERIC_FEATURE_COLUMNS))
print('X_train model input count after indexing:', len(MODEL_INPUT_COLUMNS))
print('y_train columns:', [LABEL_COLUMN])

## Step 6: Select top features with Random Forest importance

In [ ]:
selector_random_forest = RandomForestClassifier(
    n_estimators=60,  # Stable enough to rank the candidate features.
    max_depth=8,  # Allow useful interactions without overfitting the selector.
    min_samples_leaf=25,  # Avoid tiny noisy leaves.
    random_state=42,  # Keep reruns reproducible.
    n_jobs=-1,  # Use all available driver cores.
)

selector_random_forest.fit(
    train_pdf[MODEL_INPUT_COLUMNS],
    train_pdf[LABEL_COLUMN],
    sample_weight=train_pdf[WEIGHT_COLUMN],
)  # Fit the selector on the training split only.

feature_importance_pairs = sorted(zip(MODEL_INPUT_COLUMNS, [float(value) for value in selector_random_forest.feature_importances_]), key=lambda item: item[1], reverse=True)  # Rank all 18 candidate features by importance.
selected_feature_columns = [feature_name for feature_name, _ in feature_importance_pairs[:TOP_FEATURE_COUNT]]  # Keep only the top 10 features for downstream model training.
selector_importance_df = spark.createDataFrame([
    (rank, feature_name, importance_score)
    for rank, (feature_name, importance_score) in enumerate(feature_importance_pairs, start=1)
], [SELECTED_FEATURE_RANK_COLUMN, 'feature_name', 'importance_score'])  # Build a queryable importance ranking table.

del selector_random_forest  # Free the selector model now that feature importances are extracted.

print('Indexed categorical features:', INDEXED_CATEGORICAL_FEATURE_COLUMNS)
print('Numeric features:', NUMERIC_FEATURE_COLUMNS)
print('Full candidate feature vector input columns:', MODEL_INPUT_COLUMNS)
print('Selected top feature columns:', selected_feature_columns)
print('Selected X_train feature count:', len(selected_feature_columns))
display(selector_importance_df.orderBy(SELECTED_FEATURE_RANK_COLUMN))  # Review the full ranked feature list before downstream training.

## Step 7: Train the available tree-based models on the selected top features

In [ ]:
import joblib
import shutil

# CONFIRMED (project owner, 2026-08-06): this model is a compliance/monitoring signal, so a
# missed truly-late complaint (false negative) is treated as costlier than an over-flagged timely
# one (false positive) - a missed complaint is the kind of gap that shows up in a CFPB review; an
# over-flagged one just costs a reviewer a few extra minutes. 75% recall was chosen as the floor
# because it catches most untimely-response complaints while keeping precision (~14-16%) high
# enough to still function as a filter rather than flagging nearly everything. Revisit this number
# if reviewer capacity changes enough to absorb a higher floor extra false positives, or if a
# missed complaint turns out to cost more than assumed here.
TARGET_RECALL_FLOOR = 0.75
CALIBRATION_METHOD = 'sigmoid'  # See fit_score_and_log_model for why isotonic was rejected.

# Off by default: a grid search multiplies training time by len(grid) * CV_FOLDS per model, and
# the hand-picked defaults below were already tuned somewhat through manual iteration. Set True
# to search a small grid per model (scored by AUC-PR, the right metric under this class
# imbalance) instead of using the fixed hyperparameters below.
ENABLE_HYPERPARAMETER_TUNING = False
HYPERPARAMETER_TUNING_CV_FOLDS = 3
HYPERPARAMETER_GRIDS = {
    DECISION_TREE_MODEL_NAME: {'max_depth': [6, 8, 10], 'min_samples_leaf': [25, 50, 100]},
    RANDOM_FOREST_MODEL_NAME: {'n_estimators': [50, 80, 120], 'max_depth': [8, 10, 12]},
    XGBOOST_MODEL_NAME: {'max_depth': [6, 8, 10], 'learning_rate': [0.05, 0.1, 0.2]},
}


def select_threshold_for_recall_floor(labels, probabilities, recall_floor):
    # Among thresholds achieving at least recall_floor, pick the one with the highest precision.
    # Threshold selection happens on VALIDATION only - TEST is never touched until final reporting.
    precision, recall, thresholds = precision_recall_curve(labels, probabilities)
    if len(thresholds) == 0:
        return 0.5
    precision, recall = precision[:-1], recall[:-1]  # precision_recall_curve returns one extra point with no corresponding threshold.
    feasible = recall >= recall_floor
    if not feasible.any():
        # No threshold reaches the recall floor (e.g. a weak model) - fall back to the threshold that
        # maximises recall, and warn loudly rather than silently picking an arbitrary point.
        LOGGER.warning('No threshold reached the %.0f%% recall floor; falling back to the max-recall threshold.', recall_floor * 100)
        return float(thresholds[int(np.argmax(recall))])
    best_index = int(np.argmax(np.where(feasible, precision, -1.0)))
    return float(thresholds[best_index])


def plot_reliability_curve(model_name, calibration_method, labels_raw, probabilities_raw, labels_calibrated, probabilities_calibrated):
    # Show whether calibration actually fixed the probability distortion introduced by class weighting.
    fraction_raw, mean_predicted_raw = calibration_curve(labels_raw, probabilities_raw, n_bins=10, strategy='quantile')
    fraction_cal, mean_predicted_cal = calibration_curve(labels_calibrated, probabilities_calibrated, n_bins=10, strategy='quantile')

    figure, axis = plt.subplots(figsize=(5, 5))
    axis.plot([0, 1], [0, 1], linestyle='--', color='gray', label='Perfectly calibrated')
    axis.plot(mean_predicted_raw, fraction_raw, marker='o', label='Raw (class-weighted) probabilities')
    axis.plot(mean_predicted_cal, fraction_cal, marker='o', label=f'{calibration_method.capitalize()}-calibrated probabilities')
    axis.set_xlabel('Mean predicted probability')
    axis.set_ylabel('Fraction of positives')
    axis.set_title(f'Reliability curve: {model_name}')
    axis.legend()
    plt.show()


def fit_with_optional_tuning(model_name, model):
    # Fit on TRAIN, optionally hyperparameter-tuned via cross-validated grid search. Gated behind
    # ENABLE_HYPERPARAMETER_TUNING (off by default). GridSearchCV both searches and does the final
    # fit (refit=True is the default), so no separate .fit() call is needed either way.
    X_train = train_pdf[selected_feature_columns]
    y_train = train_pdf[LABEL_COLUMN]
    sample_weight = train_pdf[WEIGHT_COLUMN]

    param_grid = HYPERPARAMETER_GRIDS.get(model_name)
    if ENABLE_HYPERPARAMETER_TUNING and param_grid:
        search = GridSearchCV(
            model, param_grid,
            scoring='average_precision',  # Matches the metric used to pick the champion model.
            cv=HYPERPARAMETER_TUNING_CV_FOLDS,
            n_jobs=-1,
        )
        search.fit(X_train, y_train, sample_weight=sample_weight)
        LOGGER.info('Tuned %s via %s-fold CV: best_params=%s, best_cv_auc_pr=%.4f', model_name, HYPERPARAMETER_TUNING_CV_FOLDS, search.best_params_, search.best_score_)
        return search.best_estimator_

    model.fit(X_train, y_train, sample_weight=sample_weight)
    return model


def fit_score_and_log_model(model_name, model):
    # 1. Fit the base estimator on TRAIN (with class weights - this is what distorts its probabilities).
    model = fit_with_optional_tuning(model_name, model)

    # 2. Split VALIDATION into a calibration fold and a threshold-selection fold, so the same rows are
    #    never used both to fit the calibrator and to pick/evaluate the threshold (that would be a mild
    #    form of leakage/optimism). TEST remains untouched throughout.
    validation_calibration_pdf, validation_threshold_pdf = train_test_split(
        validation_pdf, test_size=0.5, stratify=validation_pdf[LABEL_COLUMN], random_state=42,
    )

    # 3. Calibrate on the calibration fold only, with NO sample_weight - we want probabilities that
    #    reflect the true (unweighted) class distribution, not the training-time class-weighted one.
    # Isotonic (non-parametric) calibration degenerates with this level of imbalance and a modest
    # calibration fold - confirmed by a prior run's reliability curve collapsing to near-zero
    # probabilities everywhere. Sigmoid (Platt) calibration fits a smooth 2-parameter logistic curve
    # instead, which is far more stable when positives are this rare, at the cost of less flexibility.
    calibrated_model = CalibratedClassifierCV(estimator=model, method=CALIBRATION_METHOD, cv='prefit')
    calibrated_model.fit(validation_calibration_pdf[selected_feature_columns], validation_calibration_pdf[LABEL_COLUMN])

    threshold_probability = calibrated_model.predict_proba(validation_threshold_pdf[selected_feature_columns])[:, 1]
    validation_probability = calibrated_model.predict_proba(validation_pdf[selected_feature_columns])[:, 1]  # For reporting only.
    test_probability = calibrated_model.predict_proba(test_pdf[selected_feature_columns])[:, 1]

    # Raw probabilities for the SAME rows used to fit the calibrator, so the before/after comparison
    # in the reliability curve is apples-to-apples (not an arbitrary slice of unrelated rows).
    raw_calibration_fold_probability = model.predict_proba(validation_calibration_pdf[selected_feature_columns])[:, 1]
    plot_reliability_curve(
        model_name, CALIBRATION_METHOD,
        validation_calibration_pdf[LABEL_COLUMN], raw_calibration_fold_probability,
        validation_threshold_pdf[LABEL_COLUMN], threshold_probability,
    )

    # 4. Select the threshold on the threshold-selection fold (calibrated probabilities), targeting the
    #    recall floor rather than maximising F1 - F1 weights precision and recall equally, which is the
    #    wrong objective for a compliance signal where missed positives are the costlier error.
    selected_threshold = select_threshold_for_recall_floor(
        validation_threshold_pdf[LABEL_COLUMN], threshold_probability, TARGET_RECALL_FLOOR,
    )
    validation_auc_pr = float(average_precision_score(validation_pdf[LABEL_COLUMN], validation_probability))

    # Deliberately NOT using MLflow experiment tracking (mlflow.start_run/log_params/log_metric/
    # log_model) here. On this workspace's serverless/Spark-Connect compute it has failed in four
    # distinct ways: registered-model artifact download (RESOURCE_DOES_NOT_EXIST), a Unity Catalog
    # registration path that failed silently, a crash in log_model reading the Spark config
    # spark.mlflow.modelRegistryUri (which this compute disallows reading), and finally the same
    # crash from mlflow.start_run() itself - i.e. no MLflow call here is reliable, not just model
    # logging. Metrics are already persisted to METRICS_TABLE (a real Delta table, queried directly
    # throughout this project) and the model artifact is written to a Volume with joblib below,
    # neither of which touches the MLflow client at all.
    serving_metadata = {
        'selected_feature_columns': selected_feature_columns,
        'selected_threshold': selected_threshold,
        'categorical_feature_columns': CATEGORICAL_FEATURE_COLUMNS,
        'numeric_feature_columns': NUMERIC_FEATURE_COLUMNS,
        'encoding_maps': categorical_encoding_maps,
        'global_positive_rate': global_train_positive_rate,
    }
    spark.sql(f'CREATE VOLUME IF NOT EXISTS {CATALOG}.{MONITORING_SCHEMA}.model_exports')
    volume_path = f'/Volumes/{CATALOG}/{MONITORING_SCHEMA}/model_exports/serving_metadata_{model_name}.json'
    with open(volume_path, 'w', encoding='utf-8') as serving_metadata_file:
        json.dump(serving_metadata, serving_metadata_file)
    LOGGER.info('Wrote serving metadata for %s to %s.', model_name, volume_path)

    return {
        'base_model': model,  # Raw fitted estimator - used for feature_importances_ reporting only.
        'calibrated_model': calibrated_model,
        'validation_probability': validation_probability,
        'test_probability': test_probability,
        'threshold': selected_threshold,
        'validation_auc_pr': validation_auc_pr,
    }


decision_tree_classifier = DecisionTreeClassifier(
    max_depth=8,  # Limit tree complexity for interpretability and stability.
    min_samples_leaf=50,  # Avoid tiny noisy leaves.
    random_state=42,  # Keep reruns reproducible.
)

random_forest_classifier = RandomForestClassifier(
    n_estimators=80,  # Stronger ensemble for more stable performance and importance scores.
    max_depth=10,  # Allow moderately complex interactions.
    min_samples_leaf=25,  # Avoid overly small leaves.
    random_state=42,  # Keep reruns reproducible.
    n_jobs=-1,  # Use all available driver cores.
)

training_outputs = {
    DECISION_TREE_MODEL_NAME: fit_score_and_log_model(DECISION_TREE_MODEL_NAME, decision_tree_classifier),
    RANDOM_FOREST_MODEL_NAME: fit_score_and_log_model(RANDOM_FOREST_MODEL_NAME, random_forest_classifier),
}

if XGBOOST_AVAILABLE:
    xgboost_classifier = XGBClassifier(
        objective='binary:logistic',  # Binary classification objective.
        eval_metric='aucpr',  # Precision-recall AUC is useful for imbalanced classification.
        n_estimators=120,  # Number of boosting rounds.
        max_depth=8,  # Tree depth per boosting round.
        learning_rate=0.1,  # Learning rate.
        subsample=0.8,  # Row subsampling for robustness.
        colsample_bytree=0.8,  # Column subsampling for robustness.
        random_state=42,  # Keep reruns reproducible.
        n_jobs=-1,  # Use all available driver cores.
    )
    training_outputs[XGBOOST_MODEL_NAME] = fit_score_and_log_model(XGBOOST_MODEL_NAME, xgboost_classifier)
else:
    print('XGBoost is not available in this runtime, so the notebook will compare Decision Tree and Random Forest only.')

decision_tree_validation_probability = training_outputs[DECISION_TREE_MODEL_NAME]['validation_probability']
decision_tree_test_probability = training_outputs[DECISION_TREE_MODEL_NAME]['test_probability']
decision_tree_threshold = training_outputs[DECISION_TREE_MODEL_NAME]['threshold']
decision_tree_classifier = training_outputs[DECISION_TREE_MODEL_NAME]['base_model']  # Reassign to the actually-fitted instance for importance reporting below.
random_forest_validation_probability = training_outputs[RANDOM_FOREST_MODEL_NAME]['validation_probability']
random_forest_test_probability = training_outputs[RANDOM_FOREST_MODEL_NAME]['test_probability']
random_forest_threshold = training_outputs[RANDOM_FOREST_MODEL_NAME]['threshold']
random_forest_classifier = training_outputs[RANDOM_FOREST_MODEL_NAME]['base_model']
xgboost_validation_probability = training_outputs.get(XGBOOST_MODEL_NAME, {}).get('validation_probability')
xgboost_test_probability = training_outputs.get(XGBOOST_MODEL_NAME, {}).get('test_probability')
xgboost_threshold = training_outputs.get(XGBOOST_MODEL_NAME, {}).get('threshold', 0.5)

selected_thresholds = {model_name: outputs['threshold'] for model_name, outputs in training_outputs.items()}
print('Selected per-model classification thresholds (validation-tuned for >=%.0f%% recall):' % (TARGET_RECALL_FLOOR * 100), selected_thresholds)

# End stage: pick the model with the best validation AUC-PR (the right ranking metric under this
# level of class imbalance) and write it straight to the Volume as the deployable champion, along
# with its serving metadata - both files together, atomically, so Flask always reads a matching pair.
best_model_name = max(training_outputs, key=lambda name: training_outputs[name]['validation_auc_pr'])
best_model_outputs = training_outputs[best_model_name]
print('Best model by validation AUC-PR:', best_model_name, '-> validation_auc_pr =', round(best_model_outputs['validation_auc_pr'], 4))

export_dir = f'/Volumes/{CATALOG}/{MONITORING_SCHEMA}/model_exports'
joblib.dump(best_model_outputs['calibrated_model'], f'{export_dir}/champion_model.joblib')

source_metadata_path = f'{export_dir}/serving_metadata_{best_model_name}.json'
canonical_metadata_path = f'{export_dir}/serving_metadata_champion.json'
shutil.copyfile(source_metadata_path, canonical_metadata_path)
print(f'Wrote champion model ({best_model_name}) and serving metadata to {export_dir}.')

preview_pdf = validation_pdf[['complaint_id', LABEL_COLUMN, 'product', 'state', SPLIT_COLUMN]].copy()
preview_pdf['prediction'] = (random_forest_validation_probability >= random_forest_threshold).astype(int)
preview_pdf['probability_untimely'] = random_forest_validation_probability
display(spark.createDataFrame(preview_pdf.head(20)))  # Preview scored validation examples.

## Step 8: Compare model metrics

In [ ]:
def summarize_split(model_name, split_name, labels, probabilities, threshold):
    predictions = (probabilities >= threshold).astype(int)
    labels = np.asarray(labels)

    true_positive_count = int(np.sum((labels == 1) & (predictions == 1)))
    false_positive_count = int(np.sum((labels == 0) & (predictions == 1)))
    true_negative_count = int(np.sum((labels == 0) & (predictions == 0)))
    false_negative_count = int(np.sum((labels == 1) & (predictions == 0)))

    # zero_division=0: at extreme thresholds a split can have zero predicted positives, which would
    # otherwise raise/warn rather than reporting the (correct) precision of 0 for that degenerate case.
    return {
        MODEL_NAME_COLUMN: model_name,
        'dataset_split': split_name,
        'row_count': int(len(labels)),
        'auc_roc': float(roc_auc_score(labels, probabilities)),
        'auc_pr': float(average_precision_score(labels, probabilities)),
        'precision': float(precision_score(labels, predictions, zero_division=0)),
        'recall': float(recall_score(labels, predictions, zero_division=0)),
        'f2_score': float(fbeta_score(labels, predictions, beta=2, zero_division=0)),
        'selected_threshold': float(threshold),
        'true_positive_count': true_positive_count,
        'false_positive_count': false_positive_count,
        'true_negative_count': true_negative_count,
        'false_negative_count': false_negative_count,
        'positive_label_rate': float(np.mean(labels)),
        'predicted_positive_rate': float(np.mean(predictions)),
        'feature_count': len(selected_feature_columns),
        'label_column_count': 1,
        'selected_features': ', '.join(selected_feature_columns),
    }

metrics_rows = [
    summarize_split(DECISION_TREE_MODEL_NAME, VALIDATION_SPLIT_LABEL, validation_pdf[LABEL_COLUMN], decision_tree_validation_probability, decision_tree_threshold),
    summarize_split(DECISION_TREE_MODEL_NAME, TEST_SPLIT_LABEL, test_pdf[LABEL_COLUMN], decision_tree_test_probability, decision_tree_threshold),
    summarize_split(RANDOM_FOREST_MODEL_NAME, VALIDATION_SPLIT_LABEL, validation_pdf[LABEL_COLUMN], random_forest_validation_probability, random_forest_threshold),
    summarize_split(RANDOM_FOREST_MODEL_NAME, TEST_SPLIT_LABEL, test_pdf[LABEL_COLUMN], random_forest_test_probability, random_forest_threshold),
]
if XGBOOST_AVAILABLE:
    metrics_rows.extend([
        summarize_split(XGBOOST_MODEL_NAME, VALIDATION_SPLIT_LABEL, validation_pdf[LABEL_COLUMN], xgboost_validation_probability, xgboost_threshold),
        summarize_split(XGBOOST_MODEL_NAME, TEST_SPLIT_LABEL, test_pdf[LABEL_COLUMN], xgboost_test_probability, xgboost_threshold),
    ])  # Add XGBoost metrics only when the runtime supports the library.

metrics_dataframe = spark.createDataFrame(metrics_rows).withColumn('_model_run_at', F.current_timestamp())  # Persist the metrics with a run timestamp.
display(metrics_dataframe.orderBy(MODEL_NAME_COLUMN, 'dataset_split'))  # Compare all three models across validation and test.

## Step 9: Review selection importance plus model importance from Decision Tree and Random Forest

In [ ]:
decision_tree_importance_rows = list(zip(selected_feature_columns, [float(value) for value in decision_tree_classifier.feature_importances_]))  # Pair each selected feature with its decision-tree importance.
random_forest_importance_rows = list(zip(selected_feature_columns, [float(value) for value in random_forest_classifier.feature_importances_]))  # Pair each selected feature with its random-forest importance.

decision_tree_importance_df = spark.createDataFrame([(DECISION_TREE_MODEL_NAME, feature_name, importance_score) for feature_name, importance_score in decision_tree_importance_rows], [MODEL_NAME_COLUMN, 'feature_name', 'importance_score'])  # Build a queryable decision-tree importance table.
random_forest_importance_df = spark.createDataFrame([(RANDOM_FOREST_MODEL_NAME, feature_name, importance_score) for feature_name, importance_score in random_forest_importance_rows], [MODEL_NAME_COLUMN, 'feature_name', 'importance_score'])  # Build a queryable random-forest importance table.

display(selector_importance_df.orderBy(SELECTED_FEATURE_RANK_COLUMN).limit(TOP_FEATURE_COUNT))  # Show the top features selected by the selector Random Forest.
display(decision_tree_importance_df.orderBy(F.desc('importance_score')).limit(TOP_FEATURE_COUNT))  # Show the most important decision-tree features among the selected inputs.
display(random_forest_importance_df.orderBy(F.desc('importance_score')).limit(TOP_FEATURE_COUNT))  # Show the most important random-forest features among the selected inputs.

## Step 10: Persist monitoring outputs and validate them

In [ ]:
def build_predictions_pdf(model_name, split_pdf, probabilities, threshold):
    predictions_pdf = split_pdf[[LABEL_COLUMN, SPLIT_COLUMN] + PREDICTION_OUTPUT_COLUMNS].copy()
    predictions_pdf[MODEL_NAME_COLUMN] = model_name
    predictions_pdf['prediction'] = (probabilities >= threshold).astype(int)
    predictions_pdf['predicted_probability_untimely'] = probabilities
    return predictions_pdf

predictions_parts = [
    build_predictions_pdf(DECISION_TREE_MODEL_NAME, test_pdf, decision_tree_test_probability, decision_tree_threshold),
    build_predictions_pdf(RANDOM_FOREST_MODEL_NAME, test_pdf, random_forest_test_probability, random_forest_threshold),
]
if XGBOOST_AVAILABLE:
    predictions_parts.append(build_predictions_pdf(XGBOOST_MODEL_NAME, test_pdf, xgboost_test_probability, xgboost_threshold))  # Add XGBoost predictions only when that model ran.

predictions_pdf = pd.concat(predictions_parts, ignore_index=True)
predictions_pdf = predictions_pdf[[
    MODEL_NAME_COLUMN, 'complaint_id', LABEL_COLUMN, 'prediction', 'predicted_probability_untimely',
    SPLIT_COLUMN, 'product', 'sub_product', 'company', 'state', 'issue', 'sub_issue',
    'submitted_via', 'complaint_received_date',
]]
predictions_dataframe = spark.createDataFrame(predictions_pdf).withColumn('_model_run_at', F.current_timestamp())  # Stamp every scored row with the run time.

spark.sql(f"""CREATE SCHEMA IF NOT EXISTS {CATALOG}.{MONITORING_SCHEMA} COMMENT 'Monitoring and ML evaluation outputs for consumer complaints.'""")  # Ensure the monitoring schema exists before writing outputs.

metrics_dataframe.write.format('delta').mode('overwrite').option('overwriteSchema', 'true').saveAsTable(METRICS_TABLE)  # Persist model metrics as managed Delta output.
predictions_dataframe.write.format('delta').mode('overwrite').option('overwriteSchema', 'true').saveAsTable(PREDICTIONS_TABLE)  # Persist held-out predictions as managed Delta output.

saved_metrics_dataframe = spark.table(METRICS_TABLE)  # Re-open the metrics table for post-write validation.
saved_predictions_dataframe = spark.table(PREDICTIONS_TABLE)  # Re-open the predictions table for post-write validation.

minimum_metrics_rows = 6 if XGBOOST_AVAILABLE else 4  # Expect two splits per available model.
if saved_metrics_dataframe.count() < minimum_metrics_rows:
    raise RuntimeError(f'ML output validation failed: expected at least {minimum_metrics_rows} metrics rows for the available models across validation and test.')

if saved_predictions_dataframe.count() == 0:
    raise RuntimeError('ML output validation failed: test predictions table is empty.')

display(saved_metrics_dataframe.orderBy(MODEL_NAME_COLUMN, 'dataset_split'))  # Review persisted model metrics.
display(saved_predictions_dataframe.limit(20))  # Review persisted held-out predictions.

# Placeholder drift-check hook, not implemented yet: no baseline distribution is stored or
# compared against today. Wire this up to compare this run's TRAIN split statistics (e.g.
# positive rate, per-category frequencies) against a stored historical baseline, and log/alert
# if they diverge beyond some threshold. This notebook, run top-to-bottom, is the manually
# triggered batch training entrypoint - both when run interactively and when run by the
# Databricks Job task - so this is the natural place for that check to eventually live.
print('Drift check: not yet implemented (placeholder hook).')